# 06 — Portfolio risk from a CSV

Point `PORTFOLIO_CSV` at a file of holdings and run top to bottom. You get the
portfolio's predicted risk, where it comes from, and how the model's forecasts
for these holdings have compared with what actually happened.

### CSV format

```
ticker,weight            ← or `id` (CRSP permno) instead of ticker
COST,0.08
CVX,0.07
...
```

- `weight` is optional. Without it, holdings are equal-weighted. Weights are
  rescaled to sum to 1, and negative weights (shorts) are allowed.
- Tickers are resolved **as of the forecast date**, never globally, because
  tickers get reused across firms. A ticker shared by two live securities is
  reported as ambiguous rather than guessed; use its `id` instead.
- **Only names in the 220-name sample can be priced.** `08_asset_query` (or
  `query.universe(model)`) lists the ~117 alive at the latest date. Anything else is
  listed with a reason and excluded. Nothing is dropped silently.

Everything is **annualized**. The model at date *t* prices a portfolio held from
the end of month *t* through month *t + 1*.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import db
import gics
import portfolio
import riskmath
import riskmodel

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

INK, ACCENT, MUTED, GREEN, AMBER = "#2a78d6", "#e34948", "#8a8a83", "#1baf7a", "#eda100"
GROUP_COLORS = {"market": INK, "industry": AMBER, "style": GREEN, "specific": MUTED}


def tidy(ax):
    ax.grid(alpha=.25, lw=.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


# ---- inputs -----------------------------------------------------------------
PORTFOLIO_CSV = config.PORTFOLIO_CSV    # any path, relative to this notebook
AS_OF = None                            # forecast date; None = latest
# Benchmark for active risk: the cap-weighted sample ("market") or another CSV path.
BENCHMARK = "market"

In [ ]:
engine = db.get_engine()
db.check_connection(engine)
store = riskmodel.ModelStore.from_db(engine)
model = store.at(AS_OF)
print(f"model date {model.date.date()}: {len(model.ids)} names priced, K = {len(model.factors)} factors")

---
## 1 — Load and resolve the holdings

In [ ]:
holdings, report = portfolio.load_portfolio(PORTFOLIO_CSV, store, model.date)
w = holdings["weight"]
display(holdings.join(model.meta[["sector_name", "cap"]]).sort_values("weight", ascending=False))

if BENCHMARK == "market":
    bench = model.market_weights()
    bench_name = "cap-weighted sample"
else:
    bench = portfolio.load_portfolio(BENCHMARK, store, model.date)[0]["weight"]
    bench_name = os.path.basename(BENCHMARK)

---
## 2 — Headline risk

In [ ]:
dec = model.decompose(w)
vols = model.asset_vols()
active = w.reindex(model.ids, fill_value=0.0) - bench.reindex(model.ids, fill_value=0.0)
dec_active = model.decompose(active)
beta = float(model.betas(bench).reindex(w.index) @ w)
wa = w.reindex(model.ids, fill_value=0.0)

headline = pd.Series({
    "predicted total vol": f"{dec['total_vol']:.2%}",
    "   factor vol": f"{dec['factor_vol']:.2%}",
    "   specific vol": f"{dec['specific_vol']:.2%}",
    "share of variance from factors": f"{dec['pct_factor']:.1%}",
    f"predicted beta to {bench_name}": f"{beta:.2f}",
    f"tracking error vs {bench_name}": f"{dec_active['total_vol']:.2%}",
    f"benchmark vol ({bench_name})": f"{model.vol(bench):.2%}",
    "names / effective names (1/Σw²)": f"{len(w)} / {1 / (w ** 2).sum():.1f}",
    "largest position": f"{w.max():.1%} ({holdings.loc[w.idxmax(), 'ticker']})",
    "diversification ratio Σwσ / σ_p": f"{(wa * vols['total_vol']).sum() / dec['total_vol']:.2f}",
    "1-month 95% VaR (normal)": f"{1.645 * dec['total_vol'] / np.sqrt(12):.2%}",
}, name=f"as of {model.date.date()}")
display(headline.to_frame())

---
## 3 — Where the risk comes from

Euler attribution. A factor's contribution is $y_k (F y)_k$, which sums exactly to
the factor variance, and a negative value is a hedge. The specific contribution
is $\sum_i w_i^2 \delta_i$, the only part diversification removes.

In [ ]:
g = dec["by_group"]
ga = dec_active["by_group"]
fig, ax = plt.subplots(1, 2, figsize=(13, 2.8))
for a, grp, title in ((ax[0], g, "Total risk"), (ax[1], ga, f"Active risk vs {bench_name}")):
    left = 0
    for name, row in grp.iterrows():
        v = row["pct_of_total_var"] * 100
        a.barh(0, v, left=left, color=GROUP_COLORS[name], height=.5)
        if abs(v) > 4:
            a.text(left + v / 2, 0, f"{name}\n{v:.0f}%", ha="center", va="center", color="white", fontsize=9)
        left += v
    a.set(title=title, xlabel="% of variance", yticks=[], xlim=(min(0, grp["pct_of_total_var"].min() * 100), max(100, left)))
    tidy(a)
# Legend as well as in-bar labels: the thin segments are too narrow to label.
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()]
fig.legend(handles, GROUP_COLORS.keys(), loc="lower center", ncol=4, frameon=False, fontsize=9,
           bbox_to_anchor=(.5, -.08))
plt.tight_layout()
plt.show()
display(pd.concat({"total": g, "active": ga}, axis=1).round(4))

In [ ]:
bf = dec["by_factor"].copy()
bf["benchmark exposure"] = model.exposures_of(bench)
bf["active exposure"] = bf["exposure"] - bf["benchmark exposure"]
bf.index = [f"{f} ({gics.SECTORS.get(f[4:], '')})" if f.startswith("ind_") else f for f in bf.index]
display(bf[["group", "exposure", "benchmark exposure", "active exposure", "factor_vol", "var_contrib",
            "pct_of_total_var"]].sort_values("pct_of_total_var", ascending=False).round(4))

In [ ]:
# Style exposures vs benchmark, and industry weights vs benchmark.
styles = [f for f in model.factors if riskmodel._group(f) == "style"]
inds = [f for f in model.factors if f.startswith("ind_")]
y = model.exposures_of(w)
yb = model.exposures_of(bench)

fig, ax = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1, 1.4]})
pos = np.arange(len(styles))
ax[0].barh(pos + .2, y[styles], .38, color=INK, label="portfolio")
ax[0].barh(pos - .2, yb[styles], .38, color=MUTED, label=bench_name)
ax[0].set_yticks(pos, styles)
ax[0].axvline(0, color="#1a1a19", lw=.8)
ax[0].set(title="Style exposures (σ units)", xlabel="exposure")
ax[0].legend(frameon=False, fontsize=9)
labels = [gics.SECTORS.get(f[4:], f) for f in inds]
pos = np.arange(len(inds))
ax[1].barh(pos + .2, y[inds] * 100, .38, color=INK, label="portfolio")
ax[1].barh(pos - .2, yb[inds] * 100, .38, color=MUTED, label=bench_name)
ax[1].set_yticks(pos, labels)
ax[1].set(title="Sector weights", xlabel="% of portfolio")
for a in ax:
    tidy(a)
plt.tight_layout()
plt.show()

---
## 4 — Risk by holding

**MCTR** is the change in portfolio vol per unit of added weight, and
**contribution** = weight × MCTR, which sums to total vol. A name whose
contribution share is well above its weight share is where the risk is
concentrated.

In [ ]:
ba = dec["by_asset"].join(vols[["total_vol", "specific_vol"]])
ba["weight share"] = ba["weight"] / ba["weight"].sum()
display(ba[["ticker", "conm", "sector_name", "weight", "total_vol", "specific_vol", "mctr", "vol_contrib",
            "pct_of_vol"]].round(4))

top = ba.sort_values("pct_of_vol").tail(15)
fig, ax = plt.subplots(figsize=(10, 5))
pos = np.arange(len(top))
ax.barh(pos + .2, top["pct_of_vol"] * 100, .38, color=ACCENT, label="share of risk")
ax.barh(pos - .2, top["weight share"] * 100, .38, color=MUTED, label="share of weight")
ax.set_yticks(pos, top["ticker"])
ax.set(title="Risk share vs weight share, largest risk contributors", xlabel="%")
ax.legend(frameon=False, fontsize=9)
tidy(ax)
plt.tight_layout()
plt.show()

---
## 5 — How has the model done on these holdings?

The **current** holdings are re-priced at every forecast date, keeping the names
alive then and renormalizing their weights. The realized return is the next
month's, so every point is out of sample for the risk model.

⚠️ These are today's holdings run backwards, so they are survivors by
construction. This checks whether the **risk forecast** was calibrated for this
kind of portfolio. It is not a performance backtest.

In [ ]:
R = store.realized_returns()
nxt = dict(zip(store.dates[:-1], store.dates[1:]))
hist = []
for d in store.dates:
    m = store.at(d)
    live = w.index.intersection(m.ids)
    if len(live) < 0.8 * len(w):
        continue
    wd = w.loc[live] / w.loc[live].sum()
    row = {"date": d, "names": len(live), "predicted_vol": m.vol(wd)}
    if d in nxt:
        r1 = R.loc[nxt[d]].reindex(live)
        ok = r1.notna()
        row["realized_next"] = float((wd[ok] / wd[ok].sum()) @ r1[ok]) if ok.any() else np.nan
    hist.append(row)
H = pd.DataFrame(hist).set_index("date")
H["z"] = H["realized_next"] / (H["predicted_vol"] / np.sqrt(12))
# Forward-looking: at date t, the realized vol of months t+1 .. t+12, which is
# what the forecast made at t was about.
H["realized_vol_next12"] = H["realized_next"].rolling(12).std().shift(-11) * np.sqrt(12)

b = riskmath.bias_stat(H["z"].dropna())
n = int(H["z"].notna().sum())
lo, hi = riskmath.bias_ci(n)
print(f"{n} out-of-sample months; bias statistic {b:.3f}  (95% band [{lo:.3f}, {hi:.3f}])"
      f"  {'✅ calibrated' if lo <= b <= hi else ('⚠️ risk understated' if b > hi else '⚠️ risk overstated')}")

fig, ax = plt.subplots(figsize=(13, 3.6))
ax.plot(H.index, H["predicted_vol"] * 100, color=INK, lw=2)
ax.annotate(" predicted", (H.index[-1], H["predicted_vol"].iloc[-1] * 100), color=INK, fontsize=9, va="center")
rv = H["realized_vol_next12"].dropna()
ax.plot(rv.index, rv * 100, color=ACCENT, lw=2)
ax.annotate(" realized, next 12m", (rv.index[-1], rv.iloc[-1] * 100), color=ACCENT, fontsize=9, va="center")
ax.set(title="Predicted vol at t vs realized vol over the following 12 months", ylabel="%/yr")
ax.margins(x=.12)
tidy(ax)
plt.tight_layout()
plt.show()